## Init

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
%run /Workspace/Users/axl.dxn@gmail.com/atlikon_pipeline/1_setup/utilities

## Configure widgets

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(f"catalog: {catalog}, data_source: {data_source}")

## Read from products bronze table 

In [0]:
df_silver = (
    spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
)

## Select final columns

In [0]:
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")

display(df_gold.limit(10))

## Write in gold table

In [0]:
(
    df_gold
    .write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")
)

## Sanity check of gold table

In [0]:
query = f"SELECT * FROM {catalog}.{gold_schema}.sb_dim_{data_source} LIMIT 10;"

df_check = spark.sql(query)

display(df_check.limit(10))

## Upsert operation with final dim table

In [0]:
df_sb_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM fmcg.gold.sb_dim_products;")

display(df_sb_products.limit(10))

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")

delta_table.alias("target").merge(
    source = df_sb_products.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values = {
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()